# SmolVLM-256M: Attention Implementation Check

This notebook checks which attention implementation SmolVLM-256M uses. It then compares the options on the same image and prompt.

The model can run attention in `eager` mode (plain PyTorch operations) or in `sdpa` mode (the fused PyTorch function `scaled_dot_product_attention`). The fused function can be faster and can use less memory.

The script loads the model once for each option. For each option, it prints the implementation that the text model and the vision model use. It also prints the prefill time, the graph decode time, the peak VRAM, and whether the answer is the same as the answer of the first option.

In [11]:
REPO = "./edge-ai-smol-lvm-250-optimization"
!python {REPO}/attn_check.py --image {REPO}/frame.jpg --max-len 1024

option   text/vision   prefill ms decode ms/tok peak VRAM MB  same answer
Loading weights: 100% 471/471 [00:00<00:00, 1210.78it/s]
default  sdpa/sdpa          272.8          8.13          873  True
Loading weights: 100% 471/471 [00:00<00:00, 1104.81it/s]
eager    eager/eager        392.8          6.45         2012  True
Loading weights: 100% 471/471 [00:00<00:00, 1427.10it/s]
sdpa     sdpa/sdpa          280.2          8.15          889  True


## Result

The test ran on a Colab GPU. The prompt has 875 tokens, and max_len is 1024. All options give the same answer.

| Option | Text / vision | Prefill (ms) | Decode (ms/token) | Peak VRAM (MB) |
|---|---|---|---|---|
| default | sdpa / sdpa | 272.8 | 8.13 | 873 |
| eager | eager / eager | 392.8 | 6.45 | 2012 |
| sdpa | sdpa / sdpa | 280.2 | 8.15 | 889 |

- The default option already uses `sdpa` for the text model and the vision model. To set `sdpa` changes nothing.
- `sdpa` is 1.4 times faster in prefill (280 ms and 393 ms). It also uses less peak VRAM (889 MB and 2012 MB). The `eager` option probably builds the full attention matrices, and this uses the extra memory.
- `eager` is 1.26 times faster in decode (6.45 ms and 8.15 ms per token). We do not know the cause. The `eager` run uses its own simple mask. The difference can come from the mask and not from the attention.
- For 64 new tokens, the two options take about the same time for prefill and decode (about 802 ms for `sdpa` and 806 ms for `eager`). `eager` is faster only for answers longer than about 66 tokens.
- Keep `sdpa`. The answers are short, and `sdpa` uses less VRAM.
- Next test: run `sdpa` with the same simple mask, to find the cause of the faster decode. A mix could also work: `sdpa` for prefill and `eager` for decode.

In [13]:
# sdpa+mask: sdpa attention with the same simple mask that eager uses
!python {REPO}/attn_check.py --image {REPO}/frame.jpg --max-len 1024 --impls sdpa,sdpa+mask,eager

option     text/vision   prefill ms decode ms/tok peak VRAM MB  same answer
Loading weights: 100% 471/471 [00:00<00:00, 1058.74it/s]
sdpa       sdpa/sdpa          270.4          8.14          873  True
Loading weights: 100% 471/471 [00:00<00:00, 1265.05it/s]
sdpa+mask  sdpa/sdpa          274.3          7.95          881  True
Loading weights: 100% 471/471 [00:00<00:00, 1365.90it/s]
eager      eager/eager        392.7          6.45         2020  True


## Result: mask test

The test ran on the same Colab GPU, with the same image and prompt. All options give the same answer.

| Option | Text / vision | Prefill (ms) | Decode (ms/token) | Peak VRAM (MB) |
|---|---|---|---|---|
| sdpa | sdpa / sdpa | 273.3 | 8.13 | 873 |
| sdpa+mask | sdpa / sdpa | 269.4 | 7.95 | 881 |
| eager | eager / eager | 396.1 | 6.45 | 2020 |

- The simple mask gives `sdpa` only a small gain in decode: 8.13 ms and 7.95 ms per token. The gain is 0.18 ms. The spread between runs is about 0.03 ms.
- The mask is not the main cause. `eager` is still 1.5 ms per token faster than `sdpa+mask` (6.45 ms and 7.95 ms). The attention itself is the cause. We do not know why.
- Prefill and VRAM do not change with the mask. `sdpa` uses about 270 ms and 880 MB. `eager` uses about 396 ms and 2020 MB.
- Next test: `sdpa` for prefill and `eager` for decode. This can save about 1.7 ms for each token. For 64 tokens, this is about 107 ms.